# 01 — Data Loading, Cleaning & Quality Checks

**Project:** Olist Brazilian E-Commerce — Delivery Reliability & Customer Experience  
**Stage:** Foundation only. No business conclusions are drawn in this notebook yet.

## Objective

Build a repeatable, auditable base for later analysis. This notebook:

1. Loads the nine CSV files from the original ZIP without changing the archive.
2. Profiles row counts, missing values, exact duplicates, candidate keys and relationships.
3. Applies conservative cleaning (types, whitespace, categorical casing and postal-code-prefix formatting).
4. Flags suspicious records instead of automatically deleting questionable business records.
5. Creates one deterministic geolocation lookup row per non-null postal-code prefix and tests join cardinality.
6. Writes both summary reports and row-level exception reports so an analyst can investigate specific records.
7. Runs automated quality gates and row-count reconciliation before exporting processed tables.

### Cleaning policy

**Do not remove a row just because it has missing or unusual values.** Missing delivery timestamps may be expected for canceled or in-progress orders. The review table can reuse `review_id` across orders, so `review_id` alone must not be treated as a unique key.

The raw ZIP remains the source of truth. Processed outputs are derived files and can be regenerated by rerunning this notebook. A quality check may appropriately return `REVIEW` without stopping the notebook; hard failures are reserved for violations that would make the downstream data model unsafe.


In [ ]:
from pathlib import Path
from io import BytesIO
import os
import zipfile
import json

import numpy as np
import pandas as pd
from IPython.display import display

pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 160)
pd.set_option("display.max_colwidth", 100)

# If automatic discovery fails, set this to the full path to your ZIP file.
# Example: DATA_ZIP_PATH = Path(r"C:\\Users\\YourName\\Downloads\\Brazilian E-Commerce Public Dataset.zip")
DATA_ZIP_PATH = None

# The project root is the parent folder if this notebook is run from /notebooks.
CURRENT_DIR = Path.cwd().resolve()
PROJECT_ROOT = CURRENT_DIR.parent if CURRENT_DIR.name.lower() == "notebooks" else CURRENT_DIR
RAW_DIR = PROJECT_ROOT / "data" / "raw"
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
QUALITY_DIR = PROJECT_ROOT / "outputs" / "data_quality"

for directory in (RAW_DIR, PROCESSED_DIR, QUALITY_DIR):
    directory.mkdir(parents=True, exist_ok=True)

REQUIRED_CSVS = {
    "olist_customers_dataset.csv",
    "olist_geolocation_dataset.csv",
    "olist_orders_dataset.csv",
    "olist_order_items_dataset.csv",
    "olist_order_payments_dataset.csv",
    "olist_order_reviews_dataset.csv",
    "olist_products_dataset.csv",
    "olist_sellers_dataset.csv",
    "product_category_name_translation.csv",
}


def is_olist_archive(path: Path) -> bool:
    """Return True when a ZIP contains all required Olist CSVs."""
    try:
        with zipfile.ZipFile(path) as zf:
            csv_basenames = {Path(name).name for name in zf.namelist() if name.lower().endswith(".csv")}
        return REQUIRED_CSVS.issubset(csv_basenames)
    except (OSError, zipfile.BadZipFile):
        return False


def discover_dataset_zip() -> Path:
    """Find a valid Olist ZIP, preferring data/raw over project root."""
    env_path = os.getenv("OLIST_DATA_ZIP")
    candidates = []
    if DATA_ZIP_PATH is not None:
        candidates.append(Path(DATA_ZIP_PATH).expanduser())
    if env_path:
        candidates.append(Path(env_path).expanduser())
    candidates.extend(sorted(RAW_DIR.glob("*.zip")))
    candidates.extend(sorted(PROJECT_ROOT.glob("*.zip")))

    # Helpful in this hosted session; local users should put the archive in data/raw.
    hosted_copy = Path("/mnt/data/Brazilian E-Commerce Public Dataset(1).zip")
    if hosted_copy.exists():
        candidates.append(hosted_copy)

    seen = set()
    valid = []
    for candidate in candidates:
        candidate = candidate.resolve()
        if candidate in seen or not candidate.is_file():
            continue
        seen.add(candidate)
        if is_olist_archive(candidate):
            valid.append(candidate)

    if not valid:
        raise FileNotFoundError(
            "Could not find a ZIP containing the nine Olist CSV files. "
            "Place the original ZIP in data/raw/ or set DATA_ZIP_PATH to its full path."
        )
    # If more than one match exists, explicitly configured paths / data/raw take precedence.
    if DATA_ZIP_PATH is not None:
        configured = Path(DATA_ZIP_PATH).expanduser().resolve()
        if configured in valid:
            return configured
    if env_path:
        configured = Path(env_path).expanduser().resolve()
        if configured in valid:
            return configured
    for candidate in valid:
        if candidate.parent == RAW_DIR.resolve():
            return candidate
    for candidate in valid:
        if candidate.parent == PROJECT_ROOT.resolve():
            return candidate
    if len(valid) == 1:
        return valid[0]
    raise RuntimeError(
        "Found multiple valid Olist ZIP files. Set DATA_ZIP_PATH near the top of this notebook "
        "to choose the intended file. Candidates: " + ", ".join(str(p) for p in valid)
    )

DATA_ZIP_PATH = discover_dataset_zip()
print(f"Project root : {PROJECT_ROOT}")
print(f"Dataset ZIP  : {DATA_ZIP_PATH}")
print(f"Quality out  : {QUALITY_DIR}")

## 1. Load the raw tables

Identifier columns are kept as strings. Postal-code prefixes are also loaded as strings so they can be normalized without losing leading zeros.

In [ ]:
TABLE_FILES = {
    "customers": "olist_customers_dataset.csv",
    "geolocation": "olist_geolocation_dataset.csv",
    "orders": "olist_orders_dataset.csv",
    "order_items": "olist_order_items_dataset.csv",
    "payments": "olist_order_payments_dataset.csv",
    "reviews": "olist_order_reviews_dataset.csv",
    "products": "olist_products_dataset.csv",
    "sellers": "olist_sellers_dataset.csv",
    "category_translation": "product_category_name_translation.csv",
}

DTYPE_OVERRIDES = {
    "customers": {"customer_zip_code_prefix": "string"},
    "geolocation": {"geolocation_zip_code_prefix": "string"},
    "sellers": {"seller_zip_code_prefix": "string"},
}

raw_tables = {}
with zipfile.ZipFile(DATA_ZIP_PATH) as zf:
    archive_member_by_basename = {
        Path(member).name: member
        for member in zf.namelist()
        if member.lower().endswith(".csv")
    }
    missing_files = sorted(set(TABLE_FILES.values()) - set(archive_member_by_basename))
    if missing_files:
        raise ValueError(f"The ZIP is missing expected CSV files: {missing_files}")

    for table_name, filename in TABLE_FILES.items():
        with zf.open(archive_member_by_basename[filename]) as file_obj:
            raw_tables[table_name] = pd.read_csv(
                file_obj,
                low_memory=False,
                dtype=DTYPE_OVERRIDES.get(table_name),
            )

# Work on in-memory copies. The original ZIP is never modified.
tables = {name: df.copy() for name, df in raw_tables.items()}

load_summary = pd.DataFrame([
    {"table": name, "rows": len(df), "columns": len(df.columns), "memory_mb": round(df.memory_usage(deep=True).sum() / 1024**2, 2)}
    for name, df in tables.items()
]).sort_values("table").reset_index(drop=True)

display(load_summary)
print(f"Total CSV rows loaded: {load_summary['rows'].sum():,}")

## 2. Profile the data before cleaning

We capture baseline statistics before normalization. This provides an audit trail: the cleaned data can be compared with the source, and changes are not hidden.

In [ ]:
KEY_SPECS = {
    "customers": ["customer_id"],
    "orders": ["order_id"],
    "order_items": ["order_id", "order_item_id"],
    "payments": ["order_id", "payment_sequential"],
    "reviews": ["review_id", "order_id"],
    "products": ["product_id"],
    "sellers": ["seller_id"],
    "category_translation": ["product_category_name"],
}


def make_table_profile(table_dict: dict[str, pd.DataFrame]) -> pd.DataFrame:
    rows = []
    for name, df in table_dict.items():
        rows.append({
            "table": name,
            "row_count": len(df),
            "column_count": len(df.columns),
            "exact_duplicate_rows": int(df.duplicated().sum()),
            "columns_with_missing_values": int(df.isna().any().sum()),
            "missing_cells": int(df.isna().sum().sum()),
        })
    return pd.DataFrame(rows).sort_values("table").reset_index(drop=True)


def make_column_profile(table_dict: dict[str, pd.DataFrame]) -> pd.DataFrame:
    """Use the same schema before and after cleaning; dtype is stage-agnostic."""
    rows = []
    for table_name, df in table_dict.items():
        for col in df.columns:
            series = df[col]
            non_null_examples = series.dropna().head(3).astype(str).tolist()
            rows.append({
                "table": table_name,
                "column": col,
                "dtype": str(series.dtype),
                "row_count": len(df),
                "missing_count": int(series.isna().sum()),
                "missing_pct": round(float(series.isna().mean() * 100), 3),
                "distinct_non_null": int(series.nunique(dropna=True)),
                "example_values": " | ".join(non_null_examples),
            })
    return pd.DataFrame(rows).sort_values(["table", "column"]).reset_index(drop=True)


def make_key_profile(table_dict: dict[str, pd.DataFrame]) -> pd.DataFrame:
    """Return a stable key-check schema for both pre- and post-cleaning tables."""
    rows = []
    for table_name, keys in KEY_SPECS.items():
        df = table_dict[table_name]
        null_key_rows = int(df[keys].isna().any(axis=1).sum())
        duplicate_key_excess_rows = int(df.duplicated(subset=keys, keep="first").sum())
        duplicate_key_groups = int(df.loc[df.duplicated(subset=keys, keep=False), keys].drop_duplicates().shape[0])
        rows.append({
            "table": table_name,
            "candidate_key": " + ".join(keys),
            "null_key_rows": null_key_rows,
            "duplicate_key_groups": duplicate_key_groups,
            "duplicate_key_excess_rows": duplicate_key_excess_rows,
            "passes_unique_non_null_check": null_key_rows == 0 and duplicate_key_excess_rows == 0,
        })
    return pd.DataFrame(rows).sort_values("table").reset_index(drop=True)


profile_before = make_table_profile(raw_tables)
column_profile_before = make_column_profile(raw_tables)

print("Table-level profile before cleaning")
display(profile_before)
print("Columns with the most missing values")
display(
    column_profile_before[column_profile_before["missing_count"] > 0]
    .sort_values(["missing_count", "table"], ascending=[False, True])
    .head(30)
)

profile_before.to_csv(QUALITY_DIR / "table_profile_before_cleaning.csv", index=False)
column_profile_before.to_csv(QUALITY_DIR / "column_profile_before_cleaning.csv", index=False)


## 3. Check candidate keys and table relationships

A key is not unique just because its column name ends in `_id`. The expected grain has to determine the key. The review data needs special handling: the same `review_id` can appear on more than one order in this file, but (`review_id`, `order_id`) is unique.

In [ ]:
key_profile = make_key_profile(raw_tables)
display(key_profile)
key_profile.to_csv(QUALITY_DIR / "candidate_key_checks_before_cleaning.csv", index=False)

# Special diagnostic: a review_id can be reused across different orders.
review_orders_per_id = raw_tables["reviews"].groupby("review_id", dropna=False)["order_id"].nunique(dropna=True)
review_id_reuse = pd.DataFrame([{
    "review_rows": int(len(raw_tables["reviews"])),
    "distinct_review_ids": int(raw_tables["reviews"]["review_id"].nunique(dropna=True)),
    "review_ids_attached_to_multiple_orders": int((review_orders_per_id > 1).sum()),
    "maximum_distinct_orders_per_review_id": int(review_orders_per_id.max()),
    "duplicate_review_order_pairs": int(raw_tables["reviews"].duplicated(["review_id", "order_id"]).sum()),
}])
print("Review ID reuse diagnostic — do not de-duplicate on review_id alone")
display(review_id_reuse)
review_id_reuse.to_csv(QUALITY_DIR / "review_id_reuse_diagnostic.csv", index=False)


In [ ]:
RELATIONSHIPS = [
    ("orders.customer_id", "orders", "customer_id", "customers", "customer_id"),
    ("order_items.order_id", "order_items", "order_id", "orders", "order_id"),
    ("order_items.product_id", "order_items", "product_id", "products", "product_id"),
    ("order_items.seller_id", "order_items", "seller_id", "sellers", "seller_id"),
    ("payments.order_id", "payments", "order_id", "orders", "order_id"),
    ("reviews.order_id", "reviews", "order_id", "orders", "order_id"),
    ("products.category", "products", "product_category_name", "category_translation", "product_category_name"),
    ("customers.zip_prefix", "customers", "customer_zip_code_prefix", "geolocation", "geolocation_zip_code_prefix"),
    ("sellers.zip_prefix", "sellers", "seller_zip_code_prefix", "geolocation", "geolocation_zip_code_prefix"),
]


def make_relationship_profile(table_dict: dict[str, pd.DataFrame]) -> pd.DataFrame:
    """Measure membership matches; this does not replace explicit join-cardinality tests."""
    results = []
    for label, left_table, left_col, right_table, right_col in RELATIONSHIPS:
        left = table_dict[left_table][left_col]
        right_values = set(table_dict[right_table][right_col].dropna().unique())
        unmatched_mask = left.notna() & ~left.isin(right_values)
        non_null_rows = int(left.notna().sum())
        matched_rows = int((left.notna() & left.isin(right_values)).sum())
        results.append({
            "relationship": label,
            "source_rows": len(left),
            "source_null_rows": int(left.isna().sum()),
            "source_non_null_rows": non_null_rows,
            "matched_source_rows": matched_rows,
            "unmatched_source_rows": int(unmatched_mask.sum()),
            "distinct_unmatched_values": int(left[unmatched_mask].nunique()),
            "match_rate_pct_non_null": round(100 * matched_rows / non_null_rows, 4) if non_null_rows else np.nan,
        })
    return pd.DataFrame(results)


relationships_before = make_relationship_profile(raw_tables)
display(relationships_before)
relationships_before.to_csv(QUALITY_DIR / "relationship_checks_before_cleaning.csv", index=False)


## 4. Apply conservative cleaning rules

Rules applied here:

- Trim leading/trailing whitespace; convert blank strings to missing values.
- Normalize state abbreviations to uppercase and categorical labels to lowercase.
- Parse known timestamp columns as datetime.
- Convert known numeric measures to numeric types and record failed conversions.
- Normalize postal-code prefixes to five-character strings (preserving leading zeros).
- Remove exact duplicate geolocation rows only. Do **not** remove repeated reviews, zero values, or date anomalies automatically.
- Keep missing data unless there is a justified, documented reason to impute or exclude it in a later analysis.

In [ ]:
cleaning_log = []
conversion_issues = []

# Normalize textual fields conservatively across all tables.
for table_name, df in tables.items():
    for col in df.select_dtypes(include=["object", "string"]).columns:
        cleaned_series = df[col].astype("string").str.strip()
        cleaned_series = cleaned_series.mask(cleaned_series.eq(""), pd.NA)
        changed_count = int((df[col].astype("string") != cleaned_series).fillna(False).sum())
        df[col] = cleaned_series
        if changed_count:
            cleaning_log.append({"table": table_name, "action": "trim_text_and_standardize_blank", "column": col, "rows_affected": changed_count})

# Canonical casing for selected categorical fields.
for table_name, columns in {
    "orders": ["order_status"],
    "payments": ["payment_type"],
    "products": ["product_category_name"],
    "category_translation": ["product_category_name", "product_category_name_english"],
    "customers": ["customer_city"],
    "sellers": ["seller_city"],
    "geolocation": ["geolocation_city"],
}.items():
    for col in columns:
        before = tables[table_name][col].copy()
        tables[table_name][col] = before.str.lower()
        changed_count = int((before != tables[table_name][col]).fillna(False).sum())
        if changed_count:
            cleaning_log.append({"table": table_name, "action": "lowercase_category_or_city", "column": col, "rows_affected": changed_count})

for table_name, col in [
    ("customers", "customer_state"),
    ("sellers", "seller_state"),
    ("geolocation", "geolocation_state"),
]:
    before = tables[table_name][col].copy()
    tables[table_name][col] = before.str.upper()
    changed_count = int((before != tables[table_name][col]).fillna(False).sum())
    if changed_count:
        cleaning_log.append({"table": table_name, "action": "uppercase_state_abbreviation", "column": col, "rows_affected": changed_count})

# Postal-code prefixes should be strings; zfill restores prefixes whose leading zero was not preserved in the CSV.
for table_name, col in [
    ("customers", "customer_zip_code_prefix"),
    ("sellers", "seller_zip_code_prefix"),
    ("geolocation", "geolocation_zip_code_prefix"),
]:
    before = tables[table_name][col].copy()
    tables[table_name][col] = before.astype("string").str.strip().str.zfill(5)
    changed_count = int((before.astype("string") != tables[table_name][col]).fillna(False).sum())
    cleaning_log.append({"table": table_name, "action": "normalize_zip_prefix_to_5_chars", "column": col, "rows_affected": changed_count})

DATE_COLUMNS = {
    "orders": [
        "order_purchase_timestamp", "order_approved_at", "order_delivered_carrier_date",
        "order_delivered_customer_date", "order_estimated_delivery_date",
    ],
    "order_items": ["shipping_limit_date"],
    "reviews": ["review_creation_date", "review_answer_timestamp"],
}
for table_name, columns in DATE_COLUMNS.items():
    df = tables[table_name]
    for col in columns:
        before = df[col].copy()
        parsed = pd.to_datetime(before, errors="coerce")
        failed = int((before.notna() & parsed.isna()).sum())
        conversion_issues.append({"table": table_name, "column": col, "conversion_failures": failed})
        df[col] = parsed
        cleaning_log.append({"table": table_name, "action": "parse_datetime", "column": col, "rows_affected": failed})

NUMERIC_COLUMNS = {
    "order_items": ["order_item_id", "price", "freight_value"],
    "payments": ["payment_sequential", "payment_installments", "payment_value"],
    "reviews": ["review_score"],
    "products": [
        "product_name_lenght", "product_description_lenght", "product_photos_qty",
        "product_weight_g", "product_length_cm", "product_height_cm", "product_width_cm",
    ],
    "geolocation": ["geolocation_lat", "geolocation_lng"],
}
for table_name, columns in NUMERIC_COLUMNS.items():
    df = tables[table_name]
    for col in columns:
        before = df[col].copy()
        parsed = pd.to_numeric(before, errors="coerce")
        failed = int((before.notna() & parsed.isna()).sum())
        conversion_issues.append({"table": table_name, "column": col, "conversion_failures": failed})
        df[col] = parsed
        if failed:
            cleaning_log.append({"table": table_name, "action": "numeric_conversion_failed", "column": col, "rows_affected": failed})

# Exact duplicate geolocation rows are redundant reference observations. Keep all other table rows unchanged.
geo_rows_before = len(tables["geolocation"])
tables["geolocation"] = tables["geolocation"].drop_duplicates().reset_index(drop=True)
geo_exact_duplicates_removed = geo_rows_before - len(tables["geolocation"])
cleaning_log.append({
    "table": "geolocation",
    "action": "remove_exact_duplicate_rows_only",
    "column": "ALL",
    "rows_affected": geo_exact_duplicates_removed,
})

cleaning_log_df = pd.DataFrame(cleaning_log)
conversion_issues_df = pd.DataFrame(conversion_issues)
print(f"Exact duplicate geolocation rows removed: {geo_exact_duplicates_removed:,}")
print("Numeric/date conversion failures (non-null values that could not be parsed)")
display(conversion_issues_df[conversion_issues_df["conversion_failures"] > 0] if not conversion_issues_df.empty else conversion_issues_df)
print("Cleaning log")
display(cleaning_log_df.head(40))

cleaning_log_df.to_csv(QUALITY_DIR / "cleaning_log.csv", index=False)
conversion_issues_df.to_csv(QUALITY_DIR / "type_conversion_issues.csv", index=False)

## 5. Validate date sequences and numeric anomalies

Date anomalies should be investigated, not silently corrected. In this dataset, a carrier timestamp can precede the approval timestamp for some orders. The notebook flags those rows so we can decide later whether the field is unsuitable for a specific metric.

In [ ]:
orders = tables["orders"]

DATE_SEQUENCE_CHECKS = {
    "approval_before_purchase": orders["order_approved_at"] < orders["order_purchase_timestamp"],
    "carrier_before_approval": orders["order_delivered_carrier_date"] < orders["order_approved_at"],
    "delivered_before_purchase": orders["order_delivered_customer_date"] < orders["order_purchase_timestamp"],
    "delivered_before_carrier": orders["order_delivered_customer_date"] < orders["order_delivered_carrier_date"],
    "estimated_before_purchase": orders["order_estimated_delivery_date"] < orders["order_purchase_timestamp"],
}

date_anomaly_summary = pd.DataFrame([
    {"check": name, "flagged_rows": int(mask.fillna(False).sum())}
    for name, mask in DATE_SEQUENCE_CHECKS.items()
])
display(date_anomaly_summary)

any_date_anomaly = pd.Series(False, index=orders.index)
for mask in DATE_SEQUENCE_CHECKS.values():
    any_date_anomaly |= mask.fillna(False)
flagged_columns = ["order_id", "order_status"] + list(DATE_COLUMNS["orders"])
date_anomaly_rows = orders.loc[any_date_anomaly, flagged_columns].copy()
for name, mask in DATE_SEQUENCE_CHECKS.items():
    date_anomaly_rows[name] = mask.loc[date_anomaly_rows.index].fillna(False).astype(bool)
date_anomaly_rows.to_csv(QUALITY_DIR / "order_date_anomalies.csv", index=False)
date_anomaly_summary.to_csv(QUALITY_DIR / "order_date_anomaly_summary.csv", index=False)

# Nonnegative numeric columns are checked, not auto-deleted or winsorized.
NONNEGATIVE_COLUMNS = {
    "order_items": ["price", "freight_value"],
    "payments": ["payment_installments", "payment_value"],
    "products": ["product_name_lenght", "product_description_lenght", "product_photos_qty", "product_weight_g", "product_length_cm", "product_height_cm", "product_width_cm"],
}
number_checks = []
for table_name, columns in NONNEGATIVE_COLUMNS.items():
    df = tables[table_name]
    for col in columns:
        number_checks.append({
            "table": table_name,
            "column": col,
            "negative_values": int((df[col] < 0).fillna(False).sum()),
            "zero_values": int((df[col] == 0).fillna(False).sum()),
            "missing_values": int(df[col].isna().sum()),
            "p01": df[col].quantile(0.01),
            "median": df[col].median(),
            "p99": df[col].quantile(0.99),
        })
numeric_anomaly_summary = pd.DataFrame(number_checks)
display(numeric_anomaly_summary)
numeric_anomaly_summary.to_csv(QUALITY_DIR / "numeric_anomaly_summary.csv", index=False)

## 6. Recheck keys, relationships and missing values after cleaning

This is a required control. The goal is not to make every check equal zero; the goal is to explain exceptions and ensure cleaning did not silently corrupt the data.

In [ ]:
profile_after = make_table_profile(tables)
column_profile_after = make_column_profile(tables)
key_profile_after = make_key_profile(tables)
relationships_after = make_relationship_profile(tables)

print("Table profile after cleaning")
display(profile_after)
print("Candidate key checks after cleaning")
display(key_profile_after)
print("Relationship checks after cleaning")
display(relationships_after)

profile_after.to_csv(QUALITY_DIR / "table_profile_after_cleaning.csv", index=False)
column_profile_after.to_csv(QUALITY_DIR / "column_profile_after_cleaning.csv", index=False)
key_profile_after.to_csv(QUALITY_DIR / "candidate_key_checks_after_cleaning.csv", index=False)
relationships_after.to_csv(QUALITY_DIR / "relationship_checks_after_cleaning.csv", index=False)

# Hard grain checks. Relationship mismatches are reported for investigation, not used to drop rows.
assert tables["orders"]["order_id"].notna().all(), "orders.order_id contains null values"
assert tables["orders"]["order_id"].is_unique, "orders.order_id is not unique"
assert not tables["order_items"].duplicated(["order_id", "order_item_id"]).any(), "Duplicate order item key found"
assert not tables["payments"].duplicated(["order_id", "payment_sequential"]).any(), "Duplicate payment key found"
assert not tables["reviews"].duplicated(["review_id", "order_id"]).any(), "Duplicate review/order pair found"
assert key_profile_after["passes_unique_non_null_check"].all(), "At least one tested candidate key failed after cleaning"
print("Core grain assertions passed. Relationship mismatches remain visible for review.")


## 7. Build a one-row-per-postal-prefix geolocation lookup

The raw geolocation table contains many observations for the same postal-code prefix. Joining it directly to orders/customers can multiply rows. We aggregate it to one row per normalized prefix, using median latitude/longitude and the most frequent city/state label. This is a lookup for later analyses, not a claim that every address in the prefix is at the centroid.

In [ ]:
def deterministic_mode(series: pd.Series):
    """Choose the most common non-null string; break ties lexicographically for reproducibility."""
    values = series.dropna().astype("string")
    if values.empty:
        return pd.NA
    counts = values.value_counts(dropna=True)
    max_count = int(counts.max())
    tied_values = sorted(str(value) for value in counts[counts == max_count].index.tolist())
    return tied_values[0]


# The lookup is built from exact-deduplicated geolocation observations.
# Median coordinates are robust to extreme points; mode labels are deterministic.
geo_source = tables["geolocation"].loc[
    tables["geolocation"]["geolocation_zip_code_prefix"].notna()
].copy()
geo_source["_coordinate_pair"] = (
    geo_source["geolocation_lat"].round(5).astype("string") + "|" + geo_source["geolocation_lng"].round(5).astype("string")
)

geolocation_lookup = (
    geo_source
    .groupby("geolocation_zip_code_prefix", as_index=False, dropna=True)
    .agg(
        geolocation_lat=("geolocation_lat", "median"),
        geolocation_lng=("geolocation_lng", "median"),
        geolocation_city=("geolocation_city", deterministic_mode),
        geolocation_state=("geolocation_state", deterministic_mode),
        source_observation_count=("geolocation_zip_code_prefix", "size"),
        distinct_city_count=("geolocation_city", "nunique"),
        distinct_state_count=("geolocation_state", "nunique"),
        distinct_coordinate_pairs=("_coordinate_pair", "nunique"),
        latitude_min=("geolocation_lat", "min"),
        latitude_max=("geolocation_lat", "max"),
        longitude_min=("geolocation_lng", "min"),
        longitude_max=("geolocation_lng", "max"),
    )
)

# Flag prefixes with contradictory labels or invalid/missing coordinate aggregates.
geolocation_lookup["has_city_label_conflict"] = geolocation_lookup["distinct_city_count"] > 1
geolocation_lookup["has_state_label_conflict"] = geolocation_lookup["distinct_state_count"] > 1
geolocation_lookup["has_missing_coordinates"] = geolocation_lookup[["geolocation_lat", "geolocation_lng"]].isna().any(axis=1)
geolocation_lookup["has_out_of_range_coordinates"] = (
    ~geolocation_lookup["geolocation_lat"].between(-90, 90)
    | ~geolocation_lookup["geolocation_lng"].between(-180, 180)
).fillna(False)

assert geolocation_lookup["geolocation_zip_code_prefix"].is_unique, "Geolocation lookup has repeated postal-code prefixes"
assert geolocation_lookup["geolocation_zip_code_prefix"].notna().all(), "Lookup contains a null postal-code prefix"

geo_quality_cols = [
    "geolocation_zip_code_prefix", "source_observation_count", "distinct_city_count", "distinct_state_count",
    "distinct_coordinate_pairs", "latitude_min", "latitude_max", "longitude_min", "longitude_max",
    "has_city_label_conflict", "has_state_label_conflict", "has_missing_coordinates", "has_out_of_range_coordinates",
]
geolocation_lookup_quality = geolocation_lookup[geo_quality_cols].copy()
geolocation_lookup_quality.to_csv(QUALITY_DIR / "geolocation_prefix_lookup_quality.csv", index=False)
geolocation_lookup.loc[
    geolocation_lookup[["has_city_label_conflict", "has_state_label_conflict", "has_missing_coordinates", "has_out_of_range_coordinates"]].any(axis=1)
].to_csv(QUALITY_DIR / "geolocation_prefix_lookup_exceptions.csv", index=False)

print(f"Clean geolocation observations: {len(tables['geolocation']):,}")
print(f"Unique non-null postal-code prefixes in lookup: {len(geolocation_lookup):,}")
print(f"Prefixes with conflicting city labels: {int(geolocation_lookup['has_city_label_conflict'].sum()):,}")
print(f"Prefixes with conflicting state labels: {int(geolocation_lookup['has_state_label_conflict'].sum()):,}")
display(geolocation_lookup.head())

# Validate many-to-one left joins and prove they do not change customer/seller row counts.
customer_geo_test = tables["customers"].merge(
    geolocation_lookup[["geolocation_zip_code_prefix", "geolocation_lat", "geolocation_lng", "geolocation_city", "geolocation_state"]],
    how="left", left_on="customer_zip_code_prefix", right_on="geolocation_zip_code_prefix",
    validate="many_to_one", indicator=True,
)
seller_geo_test = tables["sellers"].merge(
    geolocation_lookup[["geolocation_zip_code_prefix", "geolocation_lat", "geolocation_lng", "geolocation_city", "geolocation_state"]],
    how="left", left_on="seller_zip_code_prefix", right_on="geolocation_zip_code_prefix",
    validate="many_to_one", indicator=True,
)
assert len(customer_geo_test) == len(tables["customers"]), "Customer-to-geolocation join changed row count"
assert len(seller_geo_test) == len(tables["sellers"]), "Seller-to-geolocation join changed row count"
print("Join cardinality checks passed: customer and seller row counts were preserved.")


## 8. Create row-level exception reports and automated quality gates

The summary tables tell us how much of the data needs review; these row-level files show which records and values are involved. They are **investigation queues, not deletion lists**. The gate separates structural failures (`FAIL`) from source-data exceptions that deserve follow-up (`REVIEW`).


In [ ]:
# Row-level exception reports: these are investigative queues, not deletion lists.

# 1) Product categories missing from, or not represented by, the translation reference.
translation_values = set(tables["category_translation"]["product_category_name"].dropna().unique())
product_categories = tables["products"]["product_category_name"]
missing_category_mask = product_categories.isna()
unmapped_category_mask = product_categories.notna() & ~product_categories.isin(translation_values)
category_exception_mask = missing_category_mask | unmapped_category_mask
product_category_exceptions = tables["products"].loc[category_exception_mask, [
    "product_id", "product_category_name", "product_name_lenght", "product_description_lenght",
    "product_photos_qty", "product_weight_g", "product_length_cm", "product_height_cm", "product_width_cm"
]].copy()
product_category_exceptions["exception_type"] = np.where(
    product_category_exceptions["product_category_name"].isna(), "missing_category", "unmapped_category"
)
product_category_exceptions.to_csv(QUALITY_DIR / "unmatched_product_categories.csv", index=False)

# 2) Customer and seller postal-prefix exceptions. Include missing values and unmatched non-null values.
geo_prefixes = set(geolocation_lookup["geolocation_zip_code_prefix"].dropna().unique())

def make_zip_exceptions(df: pd.DataFrame, id_col: str, zip_col: str, city_col: str, state_col: str) -> pd.DataFrame:
    prefix = df[zip_col]
    missing_mask = prefix.isna()
    unmatched_mask = prefix.notna() & ~prefix.isin(geo_prefixes)
    result = df.loc[missing_mask | unmatched_mask, [id_col, zip_col, city_col, state_col]].copy()
    result["exception_type"] = np.where(result[zip_col].isna(), "missing_zip_prefix", "prefix_not_in_geolocation_lookup")
    return result

customer_zip_exceptions = make_zip_exceptions(tables["customers"], "customer_id", "customer_zip_code_prefix", "customer_city", "customer_state")
seller_zip_exceptions = make_zip_exceptions(tables["sellers"], "seller_id", "seller_zip_code_prefix", "seller_city", "seller_state")
customer_zip_exceptions.to_csv(QUALITY_DIR / "unmatched_customer_zip_prefixes.csv", index=False)
seller_zip_exceptions.to_csv(QUALITY_DIR / "unmatched_seller_zip_prefixes.csv", index=False)

# 3) Row-level numeric exceptions for non-negative numeric fields. Missing and zero values are flagged, not presumed wrong.
ROW_ID_COLUMNS = {
    "order_items": ["order_id", "order_item_id", "product_id", "seller_id"],
    "payments": ["order_id", "payment_sequential", "payment_type"],
    "products": ["product_id", "product_category_name"],
}
numeric_detail_rows = []
for table_name, columns in NONNEGATIVE_COLUMNS.items():
    df = tables[table_name]
    id_cols = [c for c in ROW_ID_COLUMNS.get(table_name, []) if c in df.columns]
    for col in columns:
        series = df[col]
        conditions = {
            "negative_value": series < 0,
            "zero_value": series == 0,
            "missing_value": series.isna(),
        }
        for anomaly_type, mask in conditions.items():
            mask = mask.fillna(False)
            if not mask.any():
                continue
            subset = df.loc[mask, id_cols + [col]].copy()
            for _, row in subset.iterrows():
                numeric_detail_rows.append({
                    "table": table_name,
                    "column": col,
                    "anomaly_type": anomaly_type,
                    "value": None if pd.isna(row[col]) else row[col],
                    "record_key": " | ".join(f"{key}={row[key]}" for key in id_cols),
                    "disposition": "investigate; do not delete or impute without a documented rule",
                })

numeric_anomaly_details = pd.DataFrame(numeric_detail_rows, columns=[
    "table", "column", "anomaly_type", "value", "record_key", "disposition"
])
numeric_anomaly_details.to_csv(QUALITY_DIR / "numeric_anomaly_details.csv", index=False)

# 4) Explicit postal-prefix format exceptions after normalization.
zip_format_rows = []
for table_name, col in [
    ("customers", "customer_zip_code_prefix"),
    ("sellers", "seller_zip_code_prefix"),
    ("geolocation", "geolocation_zip_code_prefix"),
]:
    df = tables[table_name]
    bad = df[col].notna() & ~df[col].astype("string").str.fullmatch(r"\d{5}", na=False)
    for idx in df.index[bad]:
        zip_format_rows.append({
            "table": table_name, "source_row_index": int(idx), "column": col,
            "value": str(df.at[idx, col]), "exception_type": "prefix_not_exactly_five_digits"
        })
pd.DataFrame(zip_format_rows, columns=["table", "source_row_index", "column", "value", "exception_type"]).to_csv(
    QUALITY_DIR / "postal_prefix_format_exceptions.csv", index=False
)

# 5) Row-count reconciliation: only exact duplicate geolocation rows may be removed at this stage.
reconciliation_rows = []
for name in sorted(raw_tables):
    raw_count = len(raw_tables[name])
    cleaned_count = len(tables[name])
    expected_removals = geo_exact_duplicates_removed if name == "geolocation" else 0
    passed = (raw_count - cleaned_count == expected_removals)
    reconciliation_rows.append({
        "table": name, "raw_rows": raw_count, "processed_rows": cleaned_count,
        "rows_removed": raw_count - cleaned_count, "rows_added": max(cleaned_count - raw_count, 0),
        "expected_rows_removed": expected_removals, "passes_reconciliation": passed,
    })
row_count_reconciliation = pd.DataFrame(reconciliation_rows)
row_count_reconciliation.to_csv(QUALITY_DIR / "row_count_reconciliation.csv", index=False)
assert row_count_reconciliation["passes_reconciliation"].all(), "Unexpected row-count change detected"

# 6) Quality-gate summary. REVIEW means investigate; FAIL is reserved for structural problems.
quality_gate_rows = []
def add_gate(check_name: str, status: str, observed: str, expected: str, note: str):
    quality_gate_rows.append({
        "check": check_name, "status": status, "observed": observed, "expected": expected, "note": note
    })

add_gate("row_count_reconciliation", "PASS" if row_count_reconciliation["passes_reconciliation"].all() else "FAIL",
         f"{int(row_count_reconciliation['passes_reconciliation'].sum())}/{len(row_count_reconciliation)} tables passed",
         "All tables unchanged except exact duplicate geolocation rows", "See row_count_reconciliation.csv")
key_failures = int((~key_profile_after["passes_unique_non_null_check"]).sum())
add_gate("candidate_keys", "PASS" if key_failures == 0 else "FAIL", f"{key_failures} failed key checks", "0 failed key checks", "See candidate_key_checks_after_cleaning.csv")
add_gate("geolocation_lookup_unique_prefix", "PASS" if geolocation_lookup["geolocation_zip_code_prefix"].is_unique else "FAIL",
         f"{geolocation_lookup['geolocation_zip_code_prefix'].nunique():,} unique keys for {len(geolocation_lookup):,} rows", "One row per non-null prefix", "Coordinates use the median; labels use deterministic mode")
add_gate("customer_geo_join_cardinality", "PASS" if len(customer_geo_test) == len(tables["customers"]) else "FAIL",
         f"{len(customer_geo_test):,} joined rows", f"{len(tables['customers']):,} customer rows", "Validated many-to-one left join")
add_gate("seller_geo_join_cardinality", "PASS" if len(seller_geo_test) == len(tables["sellers"]) else "FAIL",
         f"{len(seller_geo_test):,} joined rows", f"{len(tables['sellers']):,} seller rows", "Validated many-to-one left join")
relationship_issues = int(relationships_after["unmatched_source_rows"].sum())
add_gate("relationship_membership_exceptions", "REVIEW" if relationship_issues else "PASS",
         f"{relationship_issues:,} unmatched source references across tested relationships", "Investigate and document; do not drop rows", "See relationship checks and row-level unmatched reports")
date_issue_count = int(date_anomaly_summary["flagged_rows"].sum())
add_gate("date_sequence_anomalies", "REVIEW" if date_issue_count else "PASS",
         f"{date_issue_count:,} flagged check occurrences", "Investigate per affected duration metric", "An order can trip more than one rule; counts are not necessarily unique orders")
quality_gate = pd.DataFrame(quality_gate_rows)
quality_gate.to_csv(QUALITY_DIR / "data_quality_gate.csv", index=False)

print("\nRow-level exception reports written:")
for path in sorted(QUALITY_DIR.glob("unmatched_*.csv")) + sorted(QUALITY_DIR.glob("numeric_anomaly_details.csv")) + sorted(QUALITY_DIR.glob("postal_prefix_format_exceptions.csv")):
    print(f"- {path.name}: {sum(1 for _ in path.open(encoding='utf-8')) - 1:,} rows")
print("\nAutomated quality gate")
display(quality_gate)


## 9. Export the cleaned tables and quality reports

The exports are derived files. They are excluded from Git by default because some tables are large. You can set `EXPORT_CLEANED_TABLES = False` to run the quality checks without writing all processed CSVs.

In [ ]:
EXPORT_CLEANED_TABLES = True

if EXPORT_CLEANED_TABLES:
    for table_name, df in tables.items():
        # The cleaned geolocation observations are large and unsafe to join directly.
        # Export the validated one-row-per-prefix lookup below instead.
        if table_name == "geolocation":
            continue
        output_path = PROCESSED_DIR / f"{table_name}_clean.csv"
        df.to_csv(output_path, index=False)
        print(f"Saved {output_path.relative_to(PROJECT_ROOT)} ({len(df):,} rows)")

    print("Skipped full geolocation observation export; using the validated prefix lookup.")
    geo_lookup_path = PROCESSED_DIR / "geolocation_prefix_lookup.csv"
    geolocation_lookup.to_csv(geo_lookup_path, index=False)
    print(f"Saved {geo_lookup_path.relative_to(PROJECT_ROOT)} ({len(geolocation_lookup):,} rows)")

print(f"\nQuality reports saved in: {QUALITY_DIR}")
print("Before building the order-level model, review data_quality_gate.csv and all row-level exception reports.")


## 10. What to review before moving to the analytical model

Start with these files in `outputs/data_quality/`:

- `data_quality_gate.csv` and `row_count_reconciliation.csv` — structural checks and expected row changes.
- `candidate_key_checks_before_cleaning.csv` / `candidate_key_checks_after_cleaning.csv` — stable schemas, same tested keys.
- `relationship_checks_before_cleaning.csv` / `relationship_checks_after_cleaning.csv` — membership checks and match rates.
- `unmatched_product_categories.csv`, `unmatched_customer_zip_prefixes.csv`, `unmatched_seller_zip_prefixes.csv` — row-level reference-data exceptions.
- `numeric_anomaly_details.csv` and `order_date_anomalies.csv` — detailed numeric/date investigations.
- `geolocation_prefix_lookup_quality.csv` and `geolocation_prefix_lookup_exceptions.csv` — lookup ambiguity and coordinate checks.

### Decisions intentionally postponed

- Which delivery-date anomalies make individual duration metrics invalid.
- How missing product categories should be labeled in category analysis.
- Which review aggregation to use for sensitivity checks.
- How to treat canceled/unavailable orders in each KPI.
- Whether regional comparisons need distance, order mix or other controls.

These are analytical design choices, not generic cleaning rules. Do not delete rows merely to make a report show zero warnings.
